# Light Classification Interpretability
This notebook interprets the lightweight classification pipeline outputs. It trains CNN, BiLSTM, BiGRU, and Transformer models, extracts latent representations, and provides global + per-sample interpretability.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

%matplotlib inline

sys.path.insert(0, "..")
sys.path.insert(0, "../utils/model_training")
from model_utils import set_global_determinism
import config
from model_utils import create_cnn_model, create_lstm_model, create_gru_model, create_transformer_model

# Optional UMAP
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

tf.get_logger().setLevel('ERROR')
np.random.seed(0)

In [ ]:
import sys
import os
import joblib
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.feature_selection import mutual_info_classif
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, "..")
sys.path.insert(0, "../utils/model_training")
import config
from model_utils import (
    create_cnn_model, create_cnn_lf_model, 
    create_gru_model, create_gru_lf_model, create_cnn_gru_dual_model, 
    create_transformer_model, create_transformer_lf_model, create_cnn_transformer_dual_model, 
    set_global_determinism
)

exp_folder = Path(config.LAB_EXP_FOLDER)
out_subdir = "model_interpretation"
exp_paths = sorted([p for p in exp_folder.iterdir() if p.is_dir() and p.name != '.DS_Store'])

filter_key = None
filter_str = str(filter_key) 

joblib_path = exp_folder / "model_interpretation.joblib"

# 1. Check if there's an existing file
if joblib_path.exists():
    print(f"[*] Loading existing tracking file from {joblib_path}")
    data_packages = joblib.load(joblib_path)
else:
    print("[*] No existing tracking file found. Creating new one.")
    data_packages = []

for exp_path in exp_paths:
    out_dir = exp_path / out_subdir
    out_dir.mkdir(parents=True, exist_ok=True) 

    data = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    dataset_name = data["dataset_name"][0]
    
    existing_package = next((pkg for pkg in data_packages if pkg["dataset_name"] == dataset_name), None)
    
    if existing_package is not None:
        data_package = existing_package
        if filter_str not in data_package.get("model_paths", {}):
            print(f"[+] Appending new filter '{filter_str}' to existing dataset {dataset_name}.")
            data_package["model_paths"][filter_str] = {}
        else:
            print(f"[*] Found existing entry for {dataset_name} (Filter: '{filter_str}'). Checking for missing models...")
    else:
        print(f"[+] Creating new entry for dataset {dataset_name} with filter '{filter_str}'.")
        data_package = {
            "dataset_name" : dataset_name,
            "dataset" : data["dataset"][0],
            "features_df" : data["kinetic_features"][0],
            "y_well" : data["Y_well"],
            "timestamps" : data["timestamps"],
            "model_paths" : {filter_str: {}}
        }
        data_packages.append(data_package)

    # --- DATA PREPARATION ---
    Y_well = data_package["y_well"]
    if hasattr(config, "LABEL_MAPPINGS") and exp_path.name in config.LABEL_MAPPINGS:
        print(f"  [*] Applying custom target label mapping for experiment: {exp_path.name}")
        mapping = config.LABEL_MAPPINGS[exp_path.name]
        
        # Maps matching keys; falls back to the original index value if not found
        Y_well = [mapping.get(w, w) for w in Y_well]
    else:
        print(f"  [*] No custom mapping found for {exp_path.name}. Retaining default well labels.")


    encoder = LabelEncoder()
    y_full = encoder.fit_transform(Y_well)

    features_df = data_package["features_df"]

    if filter_key is None:
        mask = np.ones(len(y_full), dtype=bool)
    else:
        mask = (features_df[filter_key] == 1).fillna(False).values
        
    X_curve = data_package["dataset"][mask]
    y = y_full[mask]
    features_df_masked = features_df[mask]

    # Calculate Top 10 Kinetic Features (Required for Late Fusion)
    X_candidates = features_df_masked[config.LD_FEATURES].values
    X_candidates_clean = np.nan_to_num(X_candidates, nan=0.0, posinf=0.0, neginf=0.0)
    mi_scores = mutual_info_classif(X_candidates_clean, y, random_state=0)
    top_10_idx = np.argsort(mi_scores)[-10:][::-1]
    top_10_features = [config.LD_FEATURES[i] for i in top_10_idx]
    
    # Extract the manual feature set (X_man)
    X_man = features_df_masked[top_10_features].values
    X_man = np.nan_to_num(X_man, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    X_curve = X_curve.astype(np.float32)[..., None]  # (N, T, 1)
    print(f"\n[{dataset_name} | Filter: {filter_str}] X_curve shape: {X_curve.shape}, X_man shape: {X_man.shape}, y shape: {y.shape}")

    # Ensure split keeps curve and manual features aligned
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
    train_idx, test_idx = next(splitter.split(X_curve, y))
    
    X_train_curve, X_test_curve = X_curve[train_idx], X_curve[test_idx]

    scaler = StandardScaler()
    X_train_man = scaler.fit_transform(X_man[train_idx])
    X_test_man = scaler.transform(X_man[test_idx])

    y_train, y_test = y[train_idx], y[test_idx]

    input_size = X_train_curve.shape[1]
    input_man_size = X_train_man.shape[1]
    output_size = len(np.unique(y))

    # ---- TRAIN MODELS ----
    set_global_determinism(0)
    
    # LAZY INITIALIZATION: Use lambdas so models are only built if they actually need to be trained
    model_builders = {
        'cnn': lambda: create_cnn_model(input_size, output_size),
        'bigru': lambda: create_gru_model(input_size, output_size),
        'transformer': lambda: create_transformer_model(input_size, output_size),
        
        'cnn_lf': lambda: create_cnn_lf_model(input_size, input_man_size, output_size),
        'bigru_lf': lambda: create_gru_lf_model(input_size, input_man_size, output_size),
        'transformer_lf': lambda: create_transformer_lf_model(input_size, input_man_size, output_size),
        
        'cnn_gru_dual': lambda: create_cnn_gru_dual_model(input_size, output_size),
        'cnn_transformer_dual': lambda: create_cnn_transformer_dual_model(input_size, output_size),
    }

    epochs_map = {
        'cnn': 1000, 'cnn_lf': 1000, 
        'bigru': 500, 'bigru_lf': 500, 'cnn_gru_dual': 500,
        'transformer': 500, 'transformer_lf': 500, 'cnn_transformer_dual': 500 # Fixed key typo here
    }
    
    for name, builder_func in model_builders.items():
        model_filename = f"{name}_{filter_str}_model.keras"
        model_save_path = out_dir / model_filename

        # GRANULAR CHECKPOINT: Check if this specific model is already done
        if model_save_path.exists():
            print(f"  [-] Skipping {name}: Model already trained and saved.")
            data_package["model_paths"][filter_str][name] = str(model_save_path)
            continue

        print(f'  [+] Training {name}...')
        
        # Build the model dynamically only because we need it
        model = builder_func()
        
        if name.endswith('_lf'):
            train_inputs = [X_train_curve, X_train_man]
        else:
            train_inputs = X_train_curve

        history = model.fit(
            train_inputs, y_train,
            epochs=epochs_map[name],
            batch_size=512,
            shuffle=True,
            verbose=0,
        )
        print(f"      -> {name} done. val_acc={history.history['accuracy'][-1]:.3f}")

        model.save(model_save_path)
        data_package["model_paths"][filter_str][name] = str(model_save_path)
        
        # Save tracking file incrementally after every model finishes
        joblib.dump(data_packages, joblib_path)

        # Clear VRAM after each model completes
        tf.keras.backend.clear_session()

print(f"\n[*] All done. Tracking file updated at: {joblib_path}")